# 01. Exploratory Data Analysis: Obesity Risk Dataset & Synthetic Analysis

**Dataset:** `ObesityDataSet_raw_and_data_sinthetic.csv` (2,111 rows, 17 columns)  
**Goal:** Empirically analyze the distribution of features, demonstrate synthetic oversampling artifacts, and establish the `likely_original` held-out split.

---
### Key Questions:
1. What is the target class balance?
2. Which columns exhibit synthetic oversampling artifacts (fractional values in ordinal survey items)?
3. How many rows are genuine survey responses vs. synthetic SMOTE interpolations?


In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/ObesityDataSet_raw_and_data_sinthetic.csv")
print(f"Shape: {df.shape}")
print(f"Exact duplicates count: {df.duplicated().sum()}")
df.head()

In [ ]:
# 1. Target Class Distribution
print("Target Class Counts:")
print(df["NObeyesdad"].value_counts())

### 2. Evidence of Synthetic Oversampling
The ordinal survey questions were asked on discrete integer scales:
- `FCVC`: Frequency of consumption of vegetables (1, 2, 3)
- `NCP`: Number of main meals (1, 2, 3, 4)
- `CH2O`: Daily water consumption (1, 2, 3)
- `FAF`: Physical activity frequency (0, 1, 2, 3)
- `TUE`: Time using technology devices (0, 1, 2)

Because the dataset was artificially augmented using SMOTE (synthetic minority oversampling technique), fractional values were generated between discrete classes.


In [ ]:
ord_cols = ["FCVC", "NCP", "CH2O", "FAF", "TUE"]
fractional_fractions = {}
for col in ord_cols:
    frac_pct = (df[col] % 1 != 0).mean() * 100
    fractional_fractions[col] = frac_pct
    print(f"{col}: {frac_pct:.1f}% fractional values")

### 3. Isolating the `likely_original` Real Subset
We define `likely_original` rows as those where `Age` and all 5 ordinal survey columns (`FCVC`, `NCP`, `CH2O`, `FAF`, `TUE`) are exact integers.


In [ ]:
is_int = df["Age"] % 1 == 0
for c in ord_cols:
    is_int = is_int & (df[c] % 1 == 0)

likely_orig = df[is_int].drop_duplicates()
print(f"Likely original rows count: {len(likely_orig)} ({len(likely_orig) / len(df) * 100:.1f}%)")
print(
    f"Synthetic rows count: {len(df) - len(likely_orig)} ({(1 - len(likely_orig) / len(df)) * 100:.1f}%)"
)

### Conclusion
- The dataset is **~75.6% synthetic**.
- Training on synthetic SMOTE data can lead to over-optimistic test performance if evaluated only on synthetic data.
- **Decision:** We isolate the `likely_original` records as an independent held-out realism benchmark.
